# Identifying and Removing Duplicate Records Milestone - Interactive Notebook

**Topic:** Detecting and Removing Duplicate Records in Pandas DataFrames  
**Category:** Pandas Fundamentals - Data Quality  
**Date:** March 10, 2026

---

## Overview

This notebook teaches you to identify and remove duplicate records in Pandas DataFrames using:
- **duplicated()** - Detect duplicate rows
- **drop_duplicates()** - Remove duplicate rows
- **subset** - Specify columns to check for duplicates
- **keep** - Choose which duplicate to keep ('first', 'last', False)

### Why Duplicate Detection and Removal Matters

Duplicate data is one of the most common data quality issues. Duplicates can cause:
- **Inflated counts** and misleading summaries
- **Incorrect statistics** and biased analysis
- **Wasted resources** in computation
- **Wrong conclusions** from your data

**Detection → Understanding → Removal → Verification** is the proper workflow.

**Think of it as:** Removing noise so each row represents a unique, reliable observation.

---

## Learning Objectives

By completing this notebook, you will:
1. Understand what duplicate records are and why they occur
2. Detect duplicate rows using `duplicated()`
3. Identify duplicates across all or specific columns
4. Remove duplicates safely using `drop_duplicates()`
5. Choose which duplicate to keep (first, last, or none)
6. Verify deduplication results
7. Apply best practices for data quality

## Setup: Import Libraries

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

---

## Section 1: Understanding Duplicate Records

### What Are Duplicate Records?

Duplicate records are rows with **identical values** across:
- **ALL columns** (exact duplicates), OR
- **SPECIFIC columns** (partial duplicates)

### Common Causes:
- Data entry errors (same person entered twice)
- System glitches (records saved multiple times)
- Data merging (overlapping datasets)
- Lack of unique identifiers
- Historical snapshots

### Types of Duplicates:
1. **Exact duplicates** - All columns match
2. **Partial duplicates** - Only some columns match
3. **Intentional duplicates** - Valid repeated observations

⚠️ **Important:** Not all duplicates should be removed—understand context first!

### Creating Sample Data with Duplicates

In [ ]:
# Create sample customer purchase dataset with duplicates
np.random.seed(42)

sample_data = {
    'customer_id': [101, 102, 103, 101, 104, 105, 103, 106, 102, 107],
    'name': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', 'Eve', 'Charlie', 'Frank', 'Bob', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', 'alice@email.com', 
              'david@email.com', 'eve@email.com', 'charlie@email.com', 'frank@email.com', 
              'bob@email.com', 'grace@email.com'],
    'purchase_amount': [100, 150, 200, 100, 175, 225, 200, 130, 150, 190],
    'purchase_date': ['2026-01-15', '2026-01-16', '2026-01-17', '2026-01-15', 
                      '2026-01-18', '2026-01-19', '2026-01-17', '2026-01-20',
                      '2026-01-16', '2026-01-21']
}

df = pd.DataFrame(sample_data)
print("Sample customer purchase dataset:")
print(f"Shape: {df.shape}")
df

**Observations:**
- Customer ID 101 appears twice (rows 0 and 3) - EXACT duplicate
- Customer ID 102 appears twice (rows 1 and 8) - EXACT duplicate
- Customer ID 103 appears twice (rows 2 and 6) - EXACT duplicate
- These could be duplicate entries OR legitimate multiple purchases

---

## Section 2: Detecting Duplicate Rows

### Using duplicated() to Detect Duplicates

`duplicated()` returns a **boolean Series**:
- **True** where rows are duplicates
- **False** where rows are unique (or first occurrence)

**By default:**
- Considers ALL columns
- Marks subsequent duplicates as True (keeps 'first' occurrence)

**Syntax:** `df.duplicated()`

In [ ]:
# Detect duplicate rows
duplicate_mask = df.duplicated()
print("Boolean mask showing duplicate rows:")
print(duplicate_mask)
print()
print("Interpretation:")
print("- False = First occurrence or unique rows")
print("- True = Duplicate rows (subsequent occurrences)")

### Counting Duplicate Rows

In [ ]:
# Count duplicates
num_duplicates = df.duplicated().sum()
num_unique = len(df) - num_duplicates
pct_duplicates = (num_duplicates / len(df)) * 100

print(f"Total rows: {len(df)}")
print(f"Duplicate rows: {num_duplicates}")
print(f"Unique rows: {num_unique}")
print(f"Percentage duplicates: {pct_duplicates:.1f}%")

### Viewing Duplicate Rows

In [ ]:
# View only duplicate rows (subsequent occurrences)
print("Duplicate rows (subsequent occurrences only):")
duplicate_rows = df[df.duplicated()]
duplicate_rows

In [ ]:
# View ALL occurrences of duplicates (including first)
print("All rows that have duplicates (including first occurrences):")
all_duplicates = df[df.duplicated(keep=False)]
all_duplicates

**Key Difference:**
- `duplicated()` marks only **subsequent** duplicates
- `duplicated(keep=False)` marks **ALL** duplicate occurrences

---

## Section 3: Detecting Duplicates in Specific Columns

### Duplicates Based on Subset of Columns

Often, you want to check duplicates based on **specific columns** only.

Use the **`subset`** parameter to specify which columns to consider.

**Syntax:** `df.duplicated(subset=['col1', 'col2'])`

In [ ]:
# Find duplicates based on customer_id only
duplicate_by_id = df.duplicated(subset=['customer_id'])
num_id_duplicates = duplicate_by_id.sum()

print(f"Duplicates based on customer_id: {num_id_duplicates}")
print()
print("Rows with duplicate customer_id:")
df[duplicate_by_id]

### Duplicates Based on Multiple Columns

In [ ]:
# Find duplicates based on name AND email
duplicate_by_name_email = df.duplicated(subset=['name', 'email'])
num_name_email_dup = duplicate_by_name_email.sum()

print(f"Duplicates based on name + email: {num_name_email_dup}")
print()
print("Rows with duplicate name+email:")
df[duplicate_by_name_email]

### Understanding the 'keep' Parameter

The **`keep`** parameter controls which occurrence is marked as duplicate:
- **`keep='first'`** (default) - Mark duplicates except first occurrence
- **`keep='last'`** - Mark duplicates except last occurrence
- **`keep=False`** - Mark ALL duplicates (including first)

In [ ]:
# Compare different 'keep' options
print("keep='first' (default):")
print(df.duplicated(keep='first'))
print()

print("keep='last':")
print(df.duplicated(keep='last'))
print()

print("keep=False (all occurrences):")
print(df.duplicated(keep=False))

---

## Section 4: Removing Duplicate Records

### Using drop_duplicates() to Remove Duplicates

`drop_duplicates()` removes duplicate rows from DataFrame.

**Key parameters:**
- **subset**: Columns to consider for identifying duplicates
- **keep**: Which occurrence to keep ('first', 'last', False)
- **inplace**: Whether to modify original DataFrame (default False)

**Syntax:** `df.drop_duplicates()`  
**Returns:** New DataFrame with duplicates removed

In [ ]:
# Remove exact duplicates (all columns)
print("Original DataFrame:")
print(f"Shape: {df.shape}")
print(df)
print()

df_deduplicated = df.drop_duplicates()

print("After removing exact duplicates:")
print(f"Shape: {df_deduplicated.shape}")
print(df_deduplicated)
print()
print(f"Rows removed: {len(df) - len(df_deduplicated)}")

### Removing Duplicates Based on Specific Columns

In [ ]:
# Remove duplicates based on customer_id only
# Keep first occurrence of each customer
df_unique_customers = df.drop_duplicates(subset=['customer_id'])

print("After removing duplicates by customer_id:")
print(f"Shape: {df_unique_customers.shape}")
print(df_unique_customers)
print()
print(f"Original rows: {len(df)}")
print(f"Unique customers: {len(df_unique_customers)}")
print(f"Duplicate entries removed: {len(df) - len(df_unique_customers)}")

### Choosing Which Duplicate to Keep

In [ ]:
# Keep LAST occurrence instead of first
df_keep_last = df.drop_duplicates(subset=['customer_id'], keep='last')

print("Keeping last occurrence of each customer_id:")
print(df_keep_last)
print()
print("Notice how different rows are kept compared to keep='first'")

### Removing All Duplicates (Keep None)

In [ ]:
# Remove ALL occurrences of duplicated rows
# Use keep=False to remove both/all occurrences
df_no_duplicates = df.drop_duplicates(subset=['customer_id'], keep=False)

print("After removing ALL duplicate occurrences:")
print(df_no_duplicates)
print()
print(f"Rows remaining: {len(df_no_duplicates)}")
print("Only rows with unique customer_id values remain")

---

## Section 5: Practical Scenarios

### Scenario 1: Cleaning Customer Database

In [ ]:
# Create realistic customer data
customer_data = {
    'customer_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'name': ['Alice Smith', 'Alice Smith', 'Bob Jones', 'Charlie Brown', 'Bob Jones', 'Diana Prince'],
    'email': ['alice@email.com', 'alice@email.com', 'bob@email.com', 'charlie@email.com', 
              'bobjones@email.com', 'diana@email.com'],
    'phone': ['111-1111', '111-1111', '222-2222', '333-3333', '444-4444', '555-5555'],
    'registration_date': ['2026-01-01', '2026-01-15', '2026-01-10', '2026-01-20', 
                          '2026-02-01', '2026-02-05']
}

df_customers = pd.DataFrame(customer_data)

print("Original customer database:")
print(df_customers)
print()

# Check for duplicates based on email
email_duplicates = df_customers.duplicated(subset=['email'])
print(f"Duplicate emails found: {email_duplicates.sum()}")
print()

# Clean by keeping most recent registration (last occurrence)
df_customers_clean = df_customers.drop_duplicates(subset=['email'], keep='last')

print("Cleaned customer database (keeping most recent registration):")
print(df_customers_clean)
print()
print(f"Original: {len(df_customers)} customers")
print(f"Cleaned: {len(df_customers_clean)} customers")
print(f"Duplicates removed: {len(df_customers) - len(df_customers_clean)}")

### Scenario 2: Transaction Log Deduplication

In [ ]:
# Create transaction data with duplicates
transaction_data = {
    'transaction_id': ['TXN001', 'TXN002', 'TXN003', 'TXN001', 'TXN004', 'TXN005', 'TXN003'],
    'customer_id': [101, 102, 103, 101, 104, 105, 103],
    'amount': [50.00, 75.00, 100.00, 50.00, 60.00, 120.00, 100.00],
    'date': ['2026-03-01', '2026-03-01', '2026-03-02', '2026-03-01', 
             '2026-03-02', '2026-03-03', '2026-03-02'],
    'status': ['completed', 'completed', 'completed', 'completed', 
               'completed', 'completed', 'completed']
}

df_transactions = pd.DataFrame(transaction_data)

print("Transaction log with duplicates:")
print(df_transactions)
print()

# Check for duplicate transaction_ids
txn_duplicates = df_transactions.duplicated(subset=['transaction_id'])
print(f"Duplicate transactions found: {txn_duplicates.sum()}")
print()

print("Viewing duplicate transactions:")
print(df_transactions[df_transactions.duplicated(subset=['transaction_id'], keep=False)])
print()

# Remove duplicates keeping first occurrence
df_transactions_clean = df_transactions.drop_duplicates(subset=['transaction_id'], keep='first')

print("Cleaned transaction log:")
print(df_transactions_clean)
print()
print(f"Original: {len(df_transactions)} transactions")
print(f"Cleaned: {len(df_transactions_clean)} transactions")
print(f"Duplicates removed: {len(df_transactions) - len(df_transactions_clean)}")

---

## Section 6: Verifying Deduplication Results

### Compare Dataset Shapes

In [ ]:
# Compare before and after
print("Before and after comparison:")
print(f"Original DataFrame: {df.shape}")
print(f"Deduplicated DataFrame: {df_deduplicated.shape}")
print(f"Rows removed: {df.shape[0] - df_deduplicated.shape[0]}")
print(f"Columns (should be same): {df.shape[1]} → {df_deduplicated.shape[1]}")

### Recheck for Remaining Duplicates

In [ ]:
# Verify no duplicates remain
remaining_duplicates = df_deduplicated.duplicated().sum()
print(f"Duplicates remaining: {remaining_duplicates}")

if remaining_duplicates == 0:
    print("✓ SUCCESS: No duplicates remain")
else:
    print("⚠ WARNING: Duplicates still present")

### Verify Specific Column Uniqueness

In [ ]:
# Check if customer_id is now unique
unique_customer_ids = df_unique_customers['customer_id'].nunique()
total_rows = len(df_unique_customers)

print(f"Unique customer_ids: {unique_customer_ids}")
print(f"Total rows: {total_rows}")

if unique_customer_ids == total_rows:
    print("✓ SUCCESS: Each row has unique customer_id")
else:
    print("⚠ WARNING: Some customer_ids are still duplicated")

### Compare Record Counts

In [ ]:
# Statistical verification
print("Original dataset customer_id counts:")
print(df['customer_id'].value_counts().sort_index())
print()

print("Deduplicated dataset customer_id counts:")
print(df_unique_customers['customer_id'].value_counts().sort_index())

---

## Section 7: Best Practices and Common Pitfalls

### Best Practices ✓

1. **ALWAYS** check for duplicates before analysis
2. **Understand WHY** duplicates exist before removing them
3. **Use subset** parameter to specify relevant columns
4. **Choose keep** parameter intentionally (first/last/False)
5. **Verify results** after deduplication
6. **Document** what was removed and why
7. **Keep backup** of original data before removing duplicates
8. **Consider domain context** - some duplicates may be valid

### Common Pitfalls ✗

**Mistake 1: Not assigning result**
```python
❌ df.drop_duplicates()  # Does nothing!
✓ df = df.drop_duplicates()  # Correct
✓ df.drop_duplicates(inplace=True)  # Also correct
```

**Mistake 2: Wrong subset columns**
```python
❌ df.drop_duplicates(subset=['name'])  # May remove valid different people
✓ df.drop_duplicates(subset=['email'])  # Better unique identifier
```

**Mistake 3: Not considering timestamps**
```python
❌ df.drop_duplicates(subset=['customer_id'], keep='first')  # May keep older data
✓ df.sort_values('date').drop_duplicates(subset=['customer_id'], keep='last')  # Keeps most recent
```

### When NOT to Remove Duplicates

Duplicates are **NOT always errors**. Consider keeping them when:

1. **Time-series data**: Same entity at different times (e.g., customer purchases)
2. **Multi-level data**: Intentional repetition (e.g., students in multiple classes)
3. **Event logs**: Each row is an event occurrence (e.g., website visits)
4. **Many-to-many relationships**: Valid data structure (e.g., books and authors)

**Always understand your data structure before removing duplicates!**

---

## Practice Exercise

Create your own dataset with duplicates and practice:

In [ ]:
# Create your own practice dataset here
# Try different scenarios:
# 1. Product catalog with duplicate SKUs
# 2. Student enrollment with duplicate students
# 3. Sales transactions with duplicate order IDs

practice_data = {
    # Add your columns here
}

# df_practice = pd.DataFrame(practice_data)
# Practice detecting and removing duplicates

---

## Summary: Key Takeaways

### What You Learned:

1. **Understanding Duplicates**
   - What duplicate records are
   - Why they occur
   - Types of duplicates (exact vs partial)

2. **Detecting Duplicates**
   - Using `duplicated()` to identify duplicate rows
   - Counting duplicates with `sum()`
   - Viewing duplicate records
   - Understanding `keep` parameter (first/last/False)

3. **Removing Duplicates**
   - Using `drop_duplicates()` to remove duplicates
   - Specifying `subset` of columns to check
   - Choosing which occurrence to keep
   - Understanding `inplace` parameter

4. **Verifying Results**
   - Comparing dataset shapes before/after
   - Rechecking for remaining duplicates
   - Verifying column uniqueness
   - Statistical verification

5. **Best Practices**
   - Always investigate before removing
   - Use appropriate subset columns
   - Choose keep strategy intentionally
   - Verify results thoroughly
   - Document deduplication decisions

### Key Methods Summary:

```python
duplicated()                    # Detect duplicate rows
duplicated(subset=[...])        # Detect duplicates in specific columns
duplicated(keep='first')        # Mark all except first occurrence
duplicated(keep='last')         # Mark all except last occurrence
duplicated(keep=False)          # Mark all duplicate occurrences

drop_duplicates()               # Remove duplicate rows
drop_duplicates(subset=[...])   # Remove based on specific columns
drop_duplicates(keep='first')   # Keep first occurrence
drop_duplicates(keep='last')    # Keep last occurrence
drop_duplicates(keep=False)     # Remove all duplicate occurrences
drop_duplicates(inplace=True)   # Modify DataFrame in place
```

### Next Steps:

Now that you can identify and remove duplicates:

1. Practice on real datasets with duplicate records
2. Learn about missing value handling
3. Explore data validation techniques
4. Combine duplicate detection with other data quality checks
5. Build complete data cleaning pipelines

**Remember: Deduplication is a critical step in data quality assurance!**

---

## Milestone Complete! 🎉

You now have the skills to:
- ✓ Identify duplicate rows in a dataset
- ✓ Understand why duplicates occur
- ✓ Remove duplicates using appropriate methods
- ✓ Preserve important data while deduplicating
- ✓ Improve overall data quality